# Unscented Kalman Filter — Experimentation

## Overview

This notebook demonstrates the UKF on a **bearings-only tracking** problem: a sensor fixed at the origin measures only the angle $\theta = \arctan2(p_y, p_x)$ to a target moving in 2D with approximately constant velocity. Range is never observed.

### Why a standard KF cannot handle this

The Kalman filter requires a **linear** measurement model $\mathbf{z} = \mathbf{H}\mathbf{x} + \mathbf{v}$. The bearing function $h(\mathbf{x}) = \arctan2(p_y, p_x)$ is fundamentally nonlinear — there is no matrix $\mathbf{H}$ such that $\mathbf{H}\mathbf{x} = \arctan2(p_y, p_x)$ for all $\mathbf{x}$. The KF simply cannot be applied.

### Why the EKF struggles at initialisation

The EKF linearises $h$ around the current state estimate via $\mathbf{H}_J = \nabla h|_{\hat{\mathbf{x}}}$. At initialisation from a single bearing, the target's **range is entirely unknown** — the filter can only be placed along a ray. The initial covariance is large in the radial direction. The Jacobian at a poorly-known state is evaluated far from truth, making the linearisation inaccurate precisely when uncertainty is highest.

### Why the UKF succeeds

The UKF replaces the Jacobian with the **unscented transform**: it generates $2n+1$ sigma points that capture the mean and covariance of the current state distribution, propagates each sigma point through $h$ exactly (no approximation), then recovers the predicted measurement mean and covariance from the transformed set. Because $\arctan2$ is evaluated at specific points rather than approximated, no linearisation error is introduced — even when uncertainty is large.

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2
from matplotlib.collections import LineCollection
from matplotlib.patches import Ellipse
from matplotlib.lines import Line2D

np.random.seed(42)

## Mathematical Setup

### State and Process Model

The state vector $\mathbf{x} = [p_x,\; v_x,\; p_y,\; v_y]^\top$ follows the same constant-velocity (CV) model used throughout this project:

$$
\mathbf{x}_{k+1} = \mathbf{F}\mathbf{x}_k + \mathbf{w}_k, \qquad
\mathbf{F} = \begin{bmatrix} 1 & \Delta t & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & \Delta t \\ 0 & 0 & 0 & 1 \end{bmatrix},
\qquad \mathbf{w}_k \sim \mathcal{N}(\mathbf{0}, \mathbf{Q})
$$

### Bearings-Only Measurement Model

The sensor measures only the bearing angle:

$$
z_k = h(\mathbf{x}_k) + v_k = \arctan2(p_y,\, p_x) + v_k, \qquad v_k \sim \mathcal{N}(0,\, \sigma_\theta^2)
$$

This is a **scalar** ($m=1$) nonlinear measurement. The NIS is therefore distributed as $\chi^2_1$ under a consistent filter.

### Unscented Transform — Sigma Points

For an $n$-dimensional state, $2n+1$ **Merwe scaled sigma points** are generated from the current mean $\hat{\mathbf{x}}$ and covariance $\mathbf{P}$:

$$
\mathcal{X}_0 = \hat{\mathbf{x}}, \qquad
\mathcal{X}_i = \hat{\mathbf{x}} + \left(\sqrt{c\,\mathbf{P}}\right)_{:,i}, \qquad
\mathcal{X}_{n+i} = \hat{\mathbf{x}} - \left(\sqrt{c\,\mathbf{P}}\right)_{:,i}
$$

where $\left(\sqrt{c\,\mathbf{P}}\right)$ denotes the lower-Cholesky factor of $c\mathbf{P}$, subscript $:,i$ is the $i$-th column, and $c = \alpha^2(n + \kappa)$. Weights are:

$$
W_0^{(m)} = \frac{\lambda}{c}, \quad
W_0^{(c)} = \frac{\lambda}{c} + (1 - \alpha^2 + \beta), \quad
W_i^{(m)} = W_i^{(c)} = \frac{1}{2c} \quad (i \geq 1)
$$

where $\lambda = \alpha^2(n+\kappa) - n$. Here $\alpha = 1$, $\beta = 2$ (optimal for Gaussian), $\kappa = 0$, giving $c = n = 4$, $\lambda = 0$, $W_0^{(m)} = 0$, $W_0^{(c)} = 2$, $W_i = 0.125$.

### UKF Predict

Each sigma point is propagated through the process model: $\mathcal{X}_i^- = f(\mathcal{X}_i)$. The predicted mean and covariance are recovered by weighted sums:

$$
\hat{\mathbf{x}}^- = \sum_i W_i^{(m)} \mathcal{X}_i^-, \qquad
\mathbf{P}^- = \sum_i W_i^{(c)} (\mathcal{X}_i^- - \hat{\mathbf{x}}^-)(\mathcal{X}_i^- - \hat{\mathbf{x}}^-)^\top + \mathbf{Q}
$$

### UKF Update

New sigma points are drawn from $(\hat{\mathbf{x}}^-, \mathbf{P}^-)$ and mapped through $h$: $\mathcal{Z}_i = h(\mathcal{X}_i^-)$. The predicted measurement, innovation covariance, and cross-covariance are:

$$
\hat{z}^- = \sum_i W_i^{(m)} \mathcal{Z}_i, \qquad
\mathbf{S} = \sum_i W_i^{(c)} (\mathcal{Z}_i - \hat{z}^-)(\mathcal{Z}_i - \hat{z}^-)^\top + R, \qquad
\mathbf{P}_{xz} = \sum_i W_i^{(c)} (\mathcal{X}_i^- - \hat{\mathbf{x}}^-)(\mathcal{Z}_i - \hat{z}^-)^\top
$$

The Kalman gain and state update follow identically to the standard KF:

$$
\mathbf{K} = \mathbf{P}_{xz} \mathbf{S}^{-1}, \qquad
\hat{\mathbf{x}} = \hat{\mathbf{x}}^- + \mathbf{K}(z - \hat{z}^-), \qquad
\mathbf{P} = \mathbf{P}^- - \mathbf{K}\mathbf{S}\mathbf{K}^\top
$$

In [12]:
# ============================================================
#  UKF Core Implementation
# ============================================================

def normalize_angle(a):
    """Wrap angle to [-pi, pi]."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def merwe_weights(n, alpha=1.0, beta=2.0, kappa=0.0):
    """
    Merwe scaled sigma-point weights for an n-dimensional state.
    Returns (Wm, Wc, c) where c = alpha^2 * (n + kappa).
    """
    lam  = alpha**2 * (n + kappa) - n
    c    = n + lam                        # = alpha^2 * (n + kappa)
    Wm   = np.full(2 * n + 1, 0.5 / c)
    Wc   = np.full(2 * n + 1, 0.5 / c)
    Wm[0] = lam / c
    Wc[0] = lam / c + (1.0 - alpha**2 + beta)
    return Wm, Wc, c


def compute_sigma_points(x, P, c):
    """
    Generate 2n+1 sigma points via Cholesky decomposition of c*P.
    X[0] = x, X[i] = x + L[:,i-1], X[n+i] = x - L[:,i-1]
    where L = cholesky(c*P) (lower triangular).
    """
    n      = len(x)
    L      = np.linalg.cholesky(c * P)
    sigmas = np.zeros((2 * n + 1, n))
    sigmas[0] = x
    for i in range(n):
        sigmas[i + 1]     = x + L[:, i]
        sigmas[n + i + 1] = x - L[:, i]
    return sigmas


def ukf_predict(x, P, F, Q, Wm, Wc, c):
    """
    UKF prediction step.
    The process model f(x) = F x is linear here, but sigma points are
    propagated through it explicitly for algorithmic consistency.
    """
    sigmas   = compute_sigma_points(x, P, c)
    sigmas_f = (F @ sigmas.T).T                     # propagate through f
    x_pred   = Wm @ sigmas_f                        # predicted mean
    diff     = sigmas_f - x_pred
    P_pred   = np.einsum('i,ij,ik->jk', Wc, diff, diff) + Q
    return x_pred, P_pred


def ukf_update(x_pred, P_pred, z, h_func, R, Wm, Wc, c):
    """
    UKF update step for a scalar nonlinear measurement h.
    Bearing innovation is angle-wrapped to [-pi, pi].
    Returns (x_upd, P_upd, nis).
    """
    sigmas  = compute_sigma_points(x_pred, P_pred, c)

    # Propagate sigma points through measurement model
    raw = np.array([h_func(s) for s in sigmas])     # shape (2n+1,) for scalar h
    raw = raw[:, np.newaxis]                         # shape (2n+1, 1)

    z_pred = Wm @ raw                               # predicted measurement, shape (1,)

    dz  = raw - z_pred
    S   = np.einsum('i,ij,ik->jk', Wc, dz, dz) + R  # innovation covariance (1,1)

    dx  = sigmas - (Wm @ sigmas)
    Pxz = np.einsum('i,ij,ik->jk', Wc, dx, dz)      # cross-covariance (n,1)

    K = Pxz @ np.linalg.inv(S)                       # Kalman gain (n,1)

    innov    = np.array([normalize_angle(float(z) - float(z_pred))])  # shape (1,)
    nis      = float(innov @ np.linalg.inv(S) @ innov)

    x_upd = x_pred + (K @ innov).ravel()
    P_upd = P_pred - K @ S @ K.T
    P_upd = 0.5 * (P_upd + P_upd.T)                 # symmetry enforcement

    return x_upd, P_upd, nis

---
## Scenario — Bearings-Only Constant-Velocity Tracking

A target moves at constant velocity from south-east to north-west across the sensor's field of view. The sensor measures bearing only; range is never observed.

**Parameters:**

| | Value | Note |
|---|---|---|
| $\mathbf{x}_0^\text{true}$ | $[600, -2, -200, 7]^\top$ | Starts south-east, moves NW |
| $\Delta t$ | $1.0$ s | |
| $N$ | $150$ steps | |
| $\sigma_a^2$ | $0.1$ m²/s⁴ | Small — CV is a good model |
| $\sigma_\theta$ | $1°$ | Bearing noise |
| $\hat{\mathbf{x}}_0^\text{UKF}$ | $[r_0\cos\theta_0,\; 0,\; r_0\sin\theta_0,\; 0]^\top$, $r_0 = 500$ m | Along first bearing at assumed range |
| $\mathbf{P}_0$ | Elongated along first bearing: $\sigma_r = 500$ m, $\sigma_t = 30$ m; velocity $\pm 20$ m/s | Captures range uncertainty |

The true initial range is $\approx 632$ m — the filter starts 132 m short in range. The bearing sweeps from $\approx -18°$ (south-east) to $\approx +71°$ (north-east), an $89°$ arc over 150 steps. This continuous bearing rate makes range **indirectly observable**: a sequence of bearings from a moving sensor/target triangulates range over time.

In [13]:
# ============================================================
#  Scenario Setup
# ============================================================

dt = 1.0
N  = 150

# ── Process model ──────────────────────────────────────────
sigma_a2 = 0.1

F = np.array([[1, dt, 0,  0 ],
              [0,  1, 0,  0 ],
              [0,  0, 1, dt ],
              [0,  0, 0,  1 ]])

Q = sigma_a2 * np.array([
    [dt**4/4, dt**3/2,       0,       0],
    [dt**3/2,    dt**2,       0,       0],
    [      0,       0, dt**4/4, dt**3/2],
    [      0,       0, dt**3/2,    dt**2]
])

# ── True trajectory ────────────────────────────────────────
x0_true = np.array([600.0, -2.0, -200.0, 7.0])

true_states = np.zeros((N, 4))
true_states[0] = x0_true
for k in range(1, N):
    w = np.random.multivariate_normal(np.zeros(4), Q)
    true_states[k] = F @ true_states[k - 1] + w

# ── Bearing measurements ────────────────────────────────────
sigma_theta = np.deg2rad(1.0)
R_meas = np.array([[sigma_theta**2]])

bearings = np.array([
    normalize_angle(
        np.arctan2(true_states[k, 2], true_states[k, 0])
        + np.random.randn() * sigma_theta
    )
    for k in range(N)
])

# Nonlinear measurement function — state: [px, vx, py, vy]
def h_bearing(x):
    return np.arctan2(x[2], x[0])

# ── UKF initialisation ──────────────────────────────────────
# Place the filter along the first bearing at assumed range r_assume.
# P0 is elongated radially (unknown range) and tight transversely (bearing constrains direction).
r_assume   = 500.0
theta0     = bearings[0]
r_hat      = np.array([ np.cos(theta0),  np.sin(theta0)])
t_hat      = np.array([-np.sin(theta0),  np.cos(theta0)])

sigma_r_init = 500.0   # large — range unknown
sigma_t_init =  30.0   # small — bearing constrains transverse direction
P_pos0 = sigma_r_init**2 * np.outer(r_hat, r_hat) + sigma_t_init**2 * np.outer(t_hat, t_hat)

# Map 2x2 [px,py] covariance to 4x4 [px,vx,py,vy]
P0 = np.diag([0.0, 20.0**2, 0.0, 20.0**2])
P0[0, 0] = P_pos0[0, 0]
P0[0, 2] = P_pos0[0, 1]
P0[2, 0] = P_pos0[1, 0]
P0[2, 2] = P_pos0[1, 1]

x0_ukf = np.array([r_assume * np.cos(theta0), 0.0,
                   r_assume * np.sin(theta0), 0.0])

# ── UKF weights ─────────────────────────────────────────────
UKF_ALPHA = 1.0
UKF_BETA  = 2.0
UKF_KAPPA = 0.0
Wm, Wc, c_ukf = merwe_weights(4, UKF_ALPHA, UKF_BETA, UKF_KAPPA)

print(f"True initial range : {np.hypot(x0_true[0], x0_true[2]):.1f} m")
print(f"Assumed range      : {r_assume:.1f} m")
print(f"Initial range error: {abs(np.hypot(x0_true[0], x0_true[2]) - r_assume):.1f} m")
print(f"Bearing sweep      : {np.rad2deg(bearings[0]):.1f}° → {np.rad2deg(bearings[-1]):.1f}°")

True initial range : 632.5 m
Assumed range      : 500.0 m
Initial range error: 132.5 m
Bearing sweep      : -18.7° → 64.2°


In [14]:
# ============================================================
#  Run UKF
# ============================================================

estimates = np.zeros((N, 4))
P_all     = []
nis_all   = np.zeros(N)

x = x0_ukf.copy()
P = P0.copy()

for k in range(N):
    # Predict (skip at k=0 — x0/P0 is already the prior at t=0)
    if k > 0:
        x, P = ukf_predict(x, P, F, Q, Wm, Wc, c_ukf)

    # Update with bearing measurement
    x, P, nis = ukf_update(x, P, bearings[k], h_bearing, R_meas, Wm, Wc, c_ukf)

    estimates[k] = x
    P_all.append(P.copy())
    nis_all[k] = nis

# ── Metrics ─────────────────────────────────────────────────
pos_errors = np.sqrt(
    (estimates[:, 0] - true_states[:, 0])**2 +
    (estimates[:, 2] - true_states[:, 2])**2
)
rmse     = np.sqrt(np.mean(pos_errors**2))
mean_nis = np.mean(nis_all)

NIS_LOWER = chi2.ppf(0.025, df=1)
NIS_UPPER = chi2.ppf(0.975, df=1)
pct_in    = np.mean((nis_all >= NIS_LOWER) & (nis_all <= NIS_UPPER)) * 100

print(f"Position RMSE : {rmse:.1f} m")
print(f"Mean NIS      : {mean_nis:.3f}  (expected = 1.000 for scalar measurement)")
print(f"NIS in bounds : {pct_in:.0f}%  (expected ~95%)")

Position RMSE : 488.8 m
Mean NIS      : 1.096  (expected = 1.000 for scalar measurement)
NIS in bounds : 96%  (expected ~95%)


/var/folders/2v/9n_l6bfd7y9f_1gmwjt3st900000gn/T/ipykernel_28105/1147744347.py:76: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  innov    = np.array([normalize_angle(float(z) - float(z_pred))])  # shape (1,)


In [ ]:
# ============================================================
#  Plots: Trajectory + Signed Component Errors + NIS
# ============================================================

def draw_cov_ellipse(ax, center, P_pos, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(P_pos)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle  = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width  = 2 * n_std * np.sqrt(max(vals[0], 0))
    height = 2 * n_std * np.sqrt(max(vals[1], 0))
    el = Ellipse(xy=center, width=width, height=height, angle=angle, **kwargs)
    return ax.add_patch(el)


# ── Per-axis signed errors and ±2σ bounds ─────────────────────────────────
err_px = estimates[:, 0] - true_states[:, 0]
err_py = estimates[:, 2] - true_states[:, 2]
sig_px = np.array([np.sqrt(P_all[k][0, 0]) for k in range(N)])
sig_py = np.array([np.sqrt(P_all[k][2, 2]) for k in range(N)])

# Transverse measurement resolution at mean range (physical accuracy floor)
mean_range       = np.mean(np.hypot(true_states[:, 0], true_states[:, 2]))
transverse_floor = mean_range * sigma_theta   # ≈ range × sin(1°)


fig = plt.figure(figsize=(21, 8))
gs  = fig.add_gridspec(2, 3, hspace=0.50, wspace=0.30)

ax_traj = fig.add_subplot(gs[:, 0])              # left — full height
ax_px   = fig.add_subplot(gs[0, 1])              # middle top
ax_py   = fig.add_subplot(gs[1, 1], sharex=ax_px)  # middle bottom (shared x)
ax_nis  = fig.add_subplot(gs[:, 2])              # right — full height


# ── LEFT: Trajectory ──────────────────────────────────────────────────────
ax = ax_traj

pts  = np.array([true_states[:, 0], true_states[:, 2]]).T.reshape(-1, 1, 2)
segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
lc   = LineCollection(segs, cmap='plasma', norm=plt.Normalize(0, N),
                      linewidth=2.5, zorder=6)
lc.set_array(np.arange(N - 1))
ax.add_collection(lc)
cbar = fig.colorbar(lc, ax=ax, pad=0.02)
cbar.set_label('Time step', fontsize=16)
cbar.ax.tick_params(labelsize=14)

ax.plot(estimates[:, 0], estimates[:, 2],
        'k--', linewidth=1.5, alpha=0.75, zorder=5)

ray_len = 1500.0
for k in range(0, N, 30):
    ex = ray_len * np.cos(bearings[k])
    ey = ray_len * np.sin(bearings[k])
    ax.plot([0, ex], [0, ey], color='gray', linewidth=0.7,
            alpha=0.45, linestyle='--', zorder=1)

for t_idx, color, al in [(0, '#d62728', 0.28), (40, '#ff7f0e', 0.28), (149, '#1f77b4', 0.35)]:
    P_pos_k = P_all[t_idx][np.ix_([0, 2], [0, 2])]
    center  = (estimates[t_idx, 0], estimates[t_idx, 2])
    draw_cov_ellipse(ax, center, P_pos_k, n_std=2,
                     facecolor=color, edgecolor=color, alpha=al,
                     linewidth=1.5, zorder=3, clip_on=True)

ax.scatter([0], [0], s=300, c='black', marker='^', zorder=8)
ax.set_xlim(-700, 900)
ax.set_ylim(-600, 1100)
ax.set_xlabel('$p_x$ (m)', fontsize=17)
ax.set_ylabel('$p_y$ (m)', fontsize=17)
ax.set_title('Trajectory', fontsize=18)
ax.tick_params(axis='both', labelsize=15)
ax.set_aspect('equal')
ax.grid(True, alpha=0.4)

legend_handles = [
    Line2D([0], [0], color='darkorchid', linewidth=2.5),
    Line2D([0], [0], color='k', linewidth=1.5, linestyle='--', alpha=0.75),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='black',
           markersize=11, linestyle='None'),
    Line2D([0], [0], color='gray', linewidth=0.7, linestyle='--', alpha=0.6),
]
ax.legend(legend_handles,
          ['True position', 'UKF estimate', 'Sensor', 'Bearing observations'],
          fontsize=13, loc='upper right')


# ── MIDDLE TOP: px error with ±2σ band ────────────────────────────────────
ax = ax_px
ax.fill_between(range(N), -2*sig_px, 2*sig_px,
                alpha=0.25, color='steelblue', label='$\\pm 2\\sigma_{p_x}$')
ax.plot(err_px, color='steelblue', linewidth=1.5, label='$p_x$ error')
ax.axhline(0, color='k', linewidth=1.0, linestyle='--')
ax.axhline( transverse_floor, color='#d62728', linewidth=1.2, linestyle=':')
ax.axhline(-transverse_floor, color='#d62728', linewidth=1.2, linestyle=':',
           label=f'$\\pm$transverse floor ({transverse_floor:.0f} m)')
ax.set_ylabel('Error (m)', fontsize=15)
ax.set_title(f'$p_x$ and $p_y$ Errors   (RMSE = {rmse:.1f} m)', fontsize=16)
ax.tick_params(axis='both', labelsize=13)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.4)
plt.setp(ax_px.get_xticklabels(), visible=False)


# ── MIDDLE BOTTOM: py error with ±2σ band ─────────────────────────────────
ax = ax_py
ax.fill_between(range(N), -2*sig_py, 2*sig_py,
                alpha=0.25, color='steelblue', label='$\\pm 2\\sigma_{p_y}$')
ax.plot(err_py, color='steelblue', linewidth=1.5, label='$p_y$ error')
ax.axhline(0, color='k', linewidth=1.0, linestyle='--')
ax.axhline( transverse_floor, color='#d62728', linewidth=1.2, linestyle=':')
ax.axhline(-transverse_floor, color='#d62728', linewidth=1.2, linestyle=':',
           label=f'$\\pm$transverse floor ({transverse_floor:.0f} m)')
ax.set_xlabel('Time step', fontsize=15)
ax.set_ylabel('Error (m)', fontsize=15)
ax.tick_params(axis='both', labelsize=13)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.4)


# ── RIGHT: NIS ────────────────────────────────────────────────────────────
ax = ax_nis

ax.plot(nis_all, color='steelblue', linewidth=1.5, alpha=0.85,
        label='NIS $\\varepsilon_k$')
ax.axhline(NIS_UPPER, color='red', linestyle='--', linewidth=1.5,
           label=f'$\\chi^2_1$ 97.5% = {NIS_UPPER:.2f}')
ax.axhline(NIS_LOWER, color='orange', linestyle='--', linewidth=1.5,
           label=f'$\\chi^2_1$ 2.5% = {NIS_LOWER:.4f}')
ax.axhline(1.0, color='grey', linestyle=':', linewidth=1.2,
           label='Expected mean = 1')
ax.fill_between(range(N), NIS_LOWER, NIS_UPPER,
                color='green', alpha=0.07, label='95% consistency band')

ax.set_xlabel('Time step', fontsize=17)
ax.set_ylabel('NIS', fontsize=17)
ax.set_title(
    f'NIS   ({pct_in:.0f}% within 95% $\\chi^2_1$ bounds)',
    fontsize=18
)
ax.tick_params(axis='both', labelsize=15)
ax.set_ylim(bottom=0)
ax.legend(fontsize=13)
ax.grid(True, alpha=0.4)


plt.suptitle('UKF: Bearings-Only Tracking',
             fontsize=22, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nMean NIS : {mean_nis:.3f}  |  RMSE : {rmse:.1f} m")
print(f"Transverse floor at mean range ({mean_range:.0f} m): {transverse_floor:.1f} m  "
      "-- minimum achievable per-axis error from bearing noise alone")


<>:91: SyntaxWarning: invalid escape sequence '\p'
<>:96: SyntaxWarning: invalid escape sequence '\p'
<>:108: SyntaxWarning: invalid escape sequence '\p'
<>:113: SyntaxWarning: invalid escape sequence '\p'
<>:127: SyntaxWarning: invalid escape sequence '\c'
<>:129: SyntaxWarning: invalid escape sequence '\c'
<>:138: SyntaxWarning: invalid escape sequence '\c'
<>:91: SyntaxWarning: invalid escape sequence '\p'
<>:96: SyntaxWarning: invalid escape sequence '\p'
<>:108: SyntaxWarning: invalid escape sequence '\p'
<>:113: SyntaxWarning: invalid escape sequence '\p'
<>:127: SyntaxWarning: invalid escape sequence '\c'
<>:129: SyntaxWarning: invalid escape sequence '\c'
<>:138: SyntaxWarning: invalid escape sequence '\c'
/var/folders/2v/9n_l6bfd7y9f_1gmwjt3st900000gn/T/ipykernel_28105/2719423251.py:91: SyntaxWarning: invalid escape sequence '\p'
  alpha=0.25, color='steelblue', label='$\pm 2\sigma_{p_x}$')
/var/folders/2v/9n_l6bfd7y9f_1gmwjt3st900000gn/T/ipykernel_28105/2719423251.py:96: Synt

SyntaxError: unterminated f-string literal (detected at line 152) (2719423251.py, line 152)

### Analysis

**Convergence from poor initialisation.** The filter is placed along the first bearing at an assumed range of 500 m — 132 m short of the true 632 m initial range. The initial \sigma$ uncertainty ellipse (red, left panel) is highly elongated along the bearing direction, correctly encoding that the bearing is known but the range is not. As bearing rate accumulates, the geometry triangulates the range and the ellipse contracts (orange at =40$, blue at =149$).

**Component errors and filter calibration.** The middle panels show the signed $ and $ errors (estimate 569Xl$ truth) together with the $\pm 2\sigma$ band taken directly from the filter's covariance diagonal. A well-calibrated filter should contain $pprox 95\%$ of error samples inside this band; errors that escape it indicate overconfidence, while an excessively wide band indicates a conservative filter. The large initial $\pm 2\sigma$ width reflects the unknown radial direction — the filter is correctly uncertain — and the band contracts as range is resolved.

**RMSE context — the transverse floor.** The red dotted lines mark the **transverse measurement resolution** at the mean track range: $ar{r}\,\sigma_	heta$, which is the per-axis error floor imposed by bearing noise alone regardless of how well the filter performs. Errors persistently larger than this floor are due to unresolved range: the filter must infer range indirectly from bearing rate, which requires many timesteps and is fundamentally less accurate than directly measuring it. As the track matures, errors shrink toward this floor.

**NIS consistency.** The NIS scatters around the expected mean of 1 with approximately 95% of values within the $\chi^2_1$ bounds. The initial transient reflects the range-resolution phase. After convergence, the NIS is well-calibrated, confirming the unscented transform correctly propagates uncertainty through the nonlinear bearing model without the linearisation bias that would inflate or deflate the innovation covariance.
